In [ ]:
import sys
import json
import numpy as np
import tensorflow as tf
import cv2
from pathlib import Path

In [ ]:
def test_tflite_f16(image_path, model_path, labels_path):
    print(f"\n--- Testing EfficientNet (.tflite float16) ---")
    try:
        with open(labels_path, 'r', encoding='utf-8') as f:
            labels_data = json.load(f)
            labels = labels_data if isinstance(labels_data, list) else labels_data.get('labels', [])
            
        interpreter = tf.lite.Interpreter(model_path=model_path)
        interpreter.allocate_tensors()
        
        input_details = interpreter.get_input_details()[0]
        output_details = interpreter.get_output_details()[0]
        
        print(f"Input shape: {input_details['shape']}, dtype: {input_details['dtype']}")
        print(f"Output shape: {output_details['shape']}, dtype: {output_details['dtype']}")
        
        input_shape = input_details['shape']
        h, w = input_shape[1:3]
        
        img = cv2.imread(image_path)
        if img is None:
            print(f"Failed to read image at {image_path}")
            return
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img, (w, h))
        
        # Поскольку у нас Float16 конвертация, вход должен быть float32
        if input_details['dtype'] == np.uint8:
            input_data = np.expand_dims(img_resized, axis=0).astype(np.uint8)
        else:
            # Встроенный слой Rescaling в модели сам поделит значения
            input_data = np.expand_dims(img_resized, axis=0).astype(np.float32)
            
        interpreter.set_tensor(input_details['index'], input_data)
        interpreter.invoke()
        
        output_data = interpreter.get_tensor(output_details['index'])
        print(f"Raw output (first 5 elements): {output_data[0][:5]}")
        
        top_idx = np.argmax(output_data[0])
        conf = output_data[0][top_idx]
        label = labels[top_idx] if top_idx < len(labels) else f"class_{top_idx}"
        print(f"Predicted: {label} with confidence {conf:.4f}")
        
    except Exception as e:
        print(f"Error testing .tflite model: {e}")

In [ ]:
image = "C:/Dev/projects/assist-eye/assets/model_testing/test_banknote_500.jpg"
tflite = "C:/Dev/projects/assist-eye/assets/models/efficient_net/banknote_classifier_float16.tflite"
labels = "C:/Dev/projects/assist-eye/assets/models/banknote_labels.json"

In [ ]:
test_tflite_f16(image, tflite, labels)